In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import xgboost as xgb
# --- display settings ---
# Show up to 20 columns and 100 rows in any DataFrame we print.
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)
# Make plots appear inline in the notebook and set a reasonable default size.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
print('All imports successful!')



In [ ]:
df = pd.read_csv('data/AA_PAX_FORECASTING.csv', index_col=0)
df.columns = [c.replace('_', ' ').title() for c in df.columns]
raw = df.copy()
print(df.shape)
df.head()


In [ ]:
df.sample(5)

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
target = 'Od Pax'

num_cols = df.select_dtypes(include='number').columns.drop(target, errors='ignore').tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print('numeric features:', len(num_cols), '| categorical features:', len(cat_cols))

In [ ]:
print(df.isnull().sum())

print(df.isnull().sum() / len(df) * 100)

In [ ]:
missing_cols = df.columns[df.isnull().any()].tolist()
print('Columns with missing values:', len(missing_cols))

for m in missing_cols:
    for c in cat_cols:
        if c != m:
            share = df[m].isnull().groupby(df[c]).mean().sort_values(ascending=False)
            print(f'{m} missing by {c}:')
            print((share * 100).round(1).head(10), end='\n\n')

In [ ]:
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
df.nunique().sort_values(ascending=False)

In [ ]:
for col in cat_cols:
    print(df[col].value_counts(dropna=False).head(15), end='\n\n')

In [ ]:
for col in cat_cols:
    df[col].value_counts().head(15).plot(kind='bar')
    plt.title(f'Rows per {col} (top 15)')
    plt.xlabel(col)
    plt.ylabel('Number of rows')
    plt.tight_layout()
    plt.show()

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df[col].dropna(), bins=50, kde=True, ax=axes[0])
    sns.boxplot(x=df[col], ax=axes[1])
    axes[0].set_title(col)
    plt.tight_layout()
    plt.show()

In [ ]:
df[num_cols + [target]].skew()

In [ ]:
values = df[target].dropna().to_numpy()

mean_val   = np.mean(values)
median_val = np.median(values)
std_val    = np.std(values)
min_val    = np.min(values)
max_val    = np.max(values)
q1         = np.percentile(values, 25)
q3         = np.percentile(values, 75)
iqr        = q3 - q1

print(f'Mean:    {mean_val:.2f}')
print(f'Median:  {median_val:.2f}')
print(f'Std dev: {std_val:.2f}')
print(f'Min:     {min_val:.2f}')
print(f'Max:     {max_val:.2f}')
print(f'Q1:      {q1:.2f}')
print(f'Q3:      {q3:.2f}')
print(f'IQR:     {iqr:.2f}')

In [ ]:
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

is_outlier = (values < lower_fence) | (values > upper_fence)

print(f'Outliers: {is_outlier.sum()} ({100 * is_outlier.mean():.1f}% of rows)')

In [ ]:
corr = df[num_cols + [target]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.show()

In [ ]:
sns.histplot(df[target], bins=50, kde=True)
plt.title('Distribution of Od Pax')
plt.xlabel('Od Pax')
plt.ylabel('Count')
plt.show()